# Final Business Challenge — Business Performance Summary

How is our business performing, and where should management focus its attention?

This notebook synthesizes Gold-layer analytics (revenue KPIs, trends, customer/product/supplier/geography performance, returns, and supply-demand balance) into actionable findings, opportunities, risks, and recommended actions.

In [0]:
%sql
-- Executive KPI Summary
SELECT
  ROUND(SUM(net_revenue), 2)                AS net_revenue,
  COUNT(DISTINCT order_key)                  AS total_orders,
  COUNT(DISTINCT customer_key)               AS active_customers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM dev.gold.fact_sales;

In [0]:
%sql
-- Annual revenue with year-over-year change
WITH annual AS (
  SELECT order_year, ROUND(SUM(net_revenue), 2) AS net_revenue
  FROM dev.gold.fact_sales
  GROUP BY 1
)
SELECT
  order_year,
  net_revenue,
  LAG(net_revenue) OVER (ORDER BY order_year)                        AS prev_year_revenue,
  ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY order_year), 2) AS yoy_abs_change,
  ROUND(100 * (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year))
        / LAG(net_revenue) OVER (ORDER BY order_year), 2)            AS yoy_pct_change
FROM annual
ORDER BY order_year;

In [0]:
%sql
-- Revenue by market segment and by region
SELECT 'BY_SEGMENT' AS view, customer_market_segment AS category,
       ROUND(SUM(net_revenue), 2) AS net_revenue,
       ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1, 2
UNION ALL
SELECT 'BY_REGION', customer_region_name,
       ROUND(SUM(net_revenue), 2),
       ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2)
FROM dev.gold.fact_sales
GROUP BY 1, 2
ORDER BY view, net_revenue DESC;

In [0]:
%sql
-- Return flag revenue split
SELECT
  return_flag,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  SUM(quantity)              AS units_sold,
  COUNT(*)                  AS order_lines,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS revenue_share_pct
FROM dev.gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Concentration: top-10 customer & supplier share + uniform discount check
WITH top10_cust AS (
  SELECT SUM(net_revenue) AS val FROM (
    SELECT SUM(net_revenue) AS net_revenue FROM dev.gold.fact_sales
    GROUP BY customer_key ORDER BY net_revenue DESC LIMIT 10
  )
),
top10_supp AS (
  SELECT SUM(net_revenue) AS val FROM (
    SELECT SUM(net_revenue) AS net_revenue FROM dev.gold.fact_sales
    GROUP BY supplier_key ORDER BY net_revenue DESC LIMIT 10
  )
),
total AS (
  SELECT SUM(net_revenue) AS val FROM dev.gold.fact_sales
)
SELECT
  'Top-10 customer share %' AS metric,
  ROUND(100 * tc.val / t.val, 2) AS value
FROM top10_cust tc CROSS JOIN total t
UNION ALL
SELECT
  'Top-10 supplier share %',
  ROUND(100 * ts.val / t.val, 2)
FROM top10_supp ts CROSS JOIN total t
UNION ALL
SELECT
  'Avg discount rate (all segments)',
  ROUND(AVG(discount_rate) * 100, 2)
FROM dev.gold.fact_sales;

In [0]:
%sql
-- Supply-demand gap by region (supplier-side revenue vs customer-side revenue)
WITH cust AS (
  SELECT customer_region_key, customer_region_name, SUM(net_revenue) AS demand
  FROM dev.gold.fact_sales GROUP BY 1, 2
),
supp AS (
  SELECT supplier_region_key, supplier_region_name, SUM(net_revenue) AS supply
  FROM dev.gold.fact_sales GROUP BY 1, 2
)
SELECT
  COALESCE(c.customer_region_name, s.supplier_region_name) AS region_name,
  ROUND(c.demand, 2)  AS customer_demand,
  ROUND(s.supply, 2)  AS supplier_supply,
  ROUND(s.supply - c.demand, 2) AS supply_minus_demand
FROM cust c
FULL OUTER JOIN supp s ON c.customer_region_key = s.supplier_region_key
ORDER BY region_name;

## Final Business Challenge — Management Summary

**Question:** *How is our business performing, and where should management focus its attention?*

All figures are drawn from `dev.gold.fact_sales` (27.3 M order lines, 1992 – 1998) unless noted. This is synthetic TPC-H benchmark data — no real-world causation is implied.

---

### Three Key Findings (data-supported)

| # | Finding | Supporting Metric | Source Query |
|---|---------|-------------------|--------------|
| 1 | **Revenue was remarkably stable at ~$165 B/yr from 1992 – 1997, then dropped 41.3 % in 1998 to $96.97 B.** | YoY change: +0.03 %, –0.09 %, –0.26 %, +0.57 %, –0.43 % for 1993 – 1997; then **–41.32 %** in 1998 | Q2 (Annual Revenue) |
| 2 | **Revenue is extremely diversified — no concentration risk.** Top-10 customers = **0.01 %** of total; top-10 suppliers = **0.03 %**; five market segments each ≈ **20 %**; five regions each ≈ **20 %** | Segments: 19.88 – 20.10 %; Regions: 19.88 – 20.09 %; Top-10 cust share 0.01 %; Top-10 supp share 0.03 % | Q3, Q5 |
| 3 | **Nearly half of all order-line revenue comes from returned items.** Non-returned (N) = 50.63 %; Returned (R) = 24.70 %; Accepted-then-returned (A) = 24.68 % — combined returns = **49.38 %** of $1.09 T revenue | N: $551.7 B (50.63 %) · R: $269.1 B (24.70 %) · A: $269.0 B (24.68 %) | Q4 |

### Two Opportunities for Growth / Operational Improvement

| # | Opportunity | Data Evidence | Source Query |
|---|-------------|---------------|--------------|
| 1 | **Close the Africa revenue gap.** Africa is the lowest-revenue region at $216.68 B (19.88 %) with the fewest active customers (99,622 vs. Europe's 100,317). A ~$2.3 B gap separates Africa from the top region (Europe, $218.95 B) | Africa: $216.68 B, 19.88 %, 99,622 customers · Europe: $218.95 B, 20.09 %, 100,317 customers | Q3 |
| 2 | **Rebalance the supply chain geographically.** America and Asia run a combined **$2.51 B supply surplus** while Europe, Middle East, and Africa face a combined **$2.51 B supply deficit** — rerouting supplier sourcing closer to demand could cut logistics cost | AMERICA +$1.88 B · ASIA +$0.63 B surplus vs. EUROPE –$0.98 B · MIDDLE EAST –$0.97 B · AFRICA –$0.57 B deficit | Q6 |

### Two Risks Requiring Further Investigation

| # | Risk | Data Evidence | Source Query |
|---|------|---------------|--------------|
| 1 | **The 1998 revenue drop (–41.3 %) may be a data-coverage artifact, not a real decline.** TPC-H data may end mid-1998, making the full-year comparison misleading. Management must confirm data completeness before reacting | 1998 revenue $96.97 B vs. 1997 $165.25 B | Q2 |
| 2 | **Uniform 5 % discount across all segments suggests a one-size-fits-all pricing strategy.** FURNITURE (20.10 % of revenue) and HOUSEHOLD (19.88 %) receive the same discount despite different revenue contributions — margin optimisation may be possible | Avg discount = 5.00 % for all five segments | Q5 |

### Recommended Management Actions

| Category | # | Recommended Action |
|----------|---|--------------------|
| Finding 1 | 1 | Verify whether 1998 data is partial-year before drawing trend conclusions; if complete, investigate root cause of the decline |
| Finding 2 | 2 | Maintain diversification strategy — low concentration is a strength; continue avoiding over-reliance on any single customer, supplier, or region |
| Finding 3 | 3 | Launch a returns-reduction programme — 49 % return rate is high; investigate return reasons by product type, supplier, and region |
| Opportunity 1 | 4 | Targeted growth investment in Africa (sales outreach, local partnerships) to close the $2.3 B gap with Europe |
| Opportunity 2 | 5 | Conduct a supplier-sourcing review to shift procurement toward deficit regions (Europe, ME, Africa) and reduce inter-regional shipping |
| Risk 1 | 6 | Validate 1998 data coverage with the data engineering team; re-run the trend analysis once a complete year is available |
| Risk 2 | 7 | Pilot differentiated discount tiers by segment for one quarter; measure margin impact before broader rollout |

---

*Assumptions / Hypotheses (not directly supported by data):*
- The 1998 drop is assumed to be a partial-year data artifact, but this has not been confirmed.
- Supply-chain rebalancing is assumed to reduce logistics costs, but actual shipping cost data is not available in TPC-H.
- Differentiated discounts are assumed to improve margin, but price-elasticity data is not available.

**Note:** All data is from the synthetic TPC-H benchmark. No real-world business causation should be inferred.